In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any, Annotated, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [13]:
class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [14]:
def chat_node(state: ChatState) -> Any:
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot_workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
chat_id = '1'

while True:
    user_message = input('Type here: ')
    if user_message.strip().lower() in ['exit', 'quit']:
        break 

    config = {'configurable': {'thread_id': chat_id}}
    response = chatbot_workflow.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI: ', response['messages'][-1].content)

AI:  Hi Abhishek! It's nice to meet you. How can I help you today?
AI:  Yes, you just told me your name is **Abhishek**.


: 